# 05. Pandas: start

Czas: ok. 15 min

**Czego się nauczysz**
- wczytać plik Excel do Pythona i obejrzeć go jak arkusz,
- wybrać jedną kolumnę albo kilka i policzyć, jakie wartości w nich siedzą,
- filtrować wiersze warunkiem (jak filtr w Excelu),
- sortować tabelę,
- zobaczyć na własne oczy, dlaczego surowe dane trzeba najpierw wyczyścić.

## 1. Wczytanie pliku Excel

Biblioteka pandas (gotowy dodatek do Pythona) służy do pracy z tabelami.
DataFrame to tabela jak arkusz w Excelu: wiersze i kolumny z nagłówkami. Zmienną z tabelą zwyczajowo nazywa się `df`.

In [ ]:
import pandas as pd   # wczytujemy bibliotekę pandas; "pd" to przyjęty skrót, wszyscy tak piszą

# read_excel czyta plik Excela i oddaje tabelę (DataFrame); ścieżka jest względem folderu kursu
# Pułapka: w VS Code otwórz CAŁY folder kursu, nie sam plik, inaczej pandas nie znajdzie danych.
df = pd.read_excel("data/przetargi_2019_2024.xlsx")

# head() pokazuje pierwsze 5 wierszy; liczby po lewej to numery wierszy (Python liczy od 0, po filtrze zostają stare numery)
# jako ostatnia linia komórki (bez print) wyświetla się jako ładna tabela
df.head()

In [ ]:
# shape = rozmiar tabeli: (liczba wierszy, liczba kolumn)
print("Rozmiar (wiersze, kolumny):", df.shape)
# len() na tabeli = liczba wierszy (na liście dawało liczbę elementów)
print("Liczba wierszy:", len(df))
# columns = nazwy kolumn; list() zamienia je na zwykłą listę, żeby ładnie się wypisały
print("Kolumny:", list(df.columns))
# tail() pokazuje ostatnie wiersze; liczba w nawiasie mówi, ile wierszy chcemy zobaczyć
df.tail(3)

## 2. Jakiego typu są kolumny

`info()` to metryczka tabeli: nazwa każdej kolumny, ile ma wypełnionych pól i jaki ma typ.
Wszystkie kolumny są tekstem, bo dane były scrapowane (automatycznie pobierane ze stron internetowych) i przyszły jako tekst. W pandas 3 tekst wyświetla się jako `str`, w starszym pandas jako `object`; znaczy to samo.

In [ ]:
# info() wypisuje podsumowanie: Non-Null Count = ile pól jest wypełnionych, Dtype = typ kolumny
df.info()
# dtypes pokazuje same typy, bez reszty metryczki; która kolumna POWINNA być liczbą, a która datą?
# To lista do naprawy w notebooku 06 (Pandas: czyszczenie danych).
# ostatnia linia "dtype: object" opisuje samą tę listę typów, nie nasze kolumny; można ją zignorować
df.dtypes

## 3. Jedna kolumna, kilka kolumn, liczenie wartości

`df["Gmina"]` to jedna kolumna (pandas nazywa ją Series). Nazwa w cudzysłowie musi być DOKŁADNIE jak w nagłówku, ze spacją i polskimi znakami, inaczej dostaniesz `KeyError` (błąd: nie ma takiej kolumny).
Kilka kolumn wybieramy listą nazw (lista = wartości po przecinku w nawiasach kwadratowych), stąd podwójne nawiasy: `df[["Gmina", "Liczba ofert"]]`.

In [ ]:
# jedna kolumna: nazwa w cudzysłowie w nawiasach kwadratowych; print, bo komórka sama pokazuje tylko ostatnią linię
print(df["Gmina"].head())
# kilka kolumn: w środku jest lista nazw ["Gmina", "Liczba ofert"], dlatego nawiasy są podwójne; tabelę zostawiamy bez print
df[["Gmina", "Liczba ofert"]].head()

In [ ]:
# unique() = różne wartości w kolumnie, każda raz; list() zamienia wynik na zwykłą listę, żeby ładnie się wypisał
print(list(df["Województwo"].unique()))
# nunique() = ILE jest różnych wartości
print("Różnych województw:", df["Województwo"].nunique())
# gmin jest naprawdę 154, a pandas widzi więcej: spacje i wielkość liter robią z jednej gminy kilka "różnych"
print("Różnych gmin:", df["Gmina"].nunique())
# value_counts() = ile razy występuje każda wartość, od najczęstszej (jak tabela przestawna z "licznikiem")
df["Województwo"].value_counts()

In [ ]:
# value_counts() na Tryb: dwa tryby, przetarg nieograniczony jest dużo częstszy niż tryb podstawowy
print(df["Tryb"].value_counts())
# wartości tekstowe są w cudzysłowie: '1' to TEKST, nie liczba, więc pandas nie policzy z tego średniej
# (pojedynczy ' w wypisie i podwójny " w kodzie to ten sam cudzysłów); nan (bez cudzysłowu) = puste pole
print(list(df["Liczba ofert"].unique()))
# value_counts() pomija puste pola; "brak danych" to tekst, który tylko udaje puste pole, więc też trzeba go naprawić
# TODO (jeśli starczy czasu): podmień "Liczba ofert" na "Okres umowy" w linii niżej. Ile różnych zapisów znaczy "24 miesiące"?
df["Liczba ofert"].value_counts()

## 4. Filtrowanie, czyli warunek na całej kolumnie

Porównanie dwóch wartości, np. `3 == 1`, daje `True` albo `False`; w pandas ten sam warunek na kolumnie sprawdza wszystkie wiersze naraz i daje kolumnę `True`/`False`, tzw. maskę.
`df[mask]` zostawia tylko wiersze z `True`, jak filtr w Excelu. Pułapki: `==` (dwa znaki), a nie `=`; `"1"` w cudzysłowie, bo kolumna jest tekstem.

In [ ]:
# maska = wynik porównania dla każdego wiersza: True, gdy województwo się zgadza, False, gdy nie
mask = df["Województwo"] == "małopolskie"
# początek maski: sama kolumna True/False, jeszcze bez danych
print(mask.head())
# df[mask] = tylko wiersze, dla których maska ma True (filtr w Excelu); len() liczy, ile ich zostało
print("Przetargów w woj. małopolskim:", len(df[mask]))
df[mask].head()

In [ ]:
# dwa warunki: & znaczy "i" (oba spełnione), | znaczy "lub" (wystarczy jeden)
# Pułapka 1: na kolumnach NIE działają słowa and/or (te są dla pojedynczych wartości), tylko znaki & i |.
# Pułapka 2: każdy warunek MUSI być w nawiasach, inaczej pandas pomiesza kolejność i wyrzuci błąd.
both = df[(df["Województwo"] == "małopolskie") & (df["Liczba ofert"] == "1")]
either = df[(df["Województwo"] == "małopolskie") | (df["Województwo"] == "śląskie")]
print("Małopolskie I jedna oferta:", len(both))
print("Małopolskie LUB śląskie:", len(either))

**Twoja kolej:** własny filtr z dwoma warunkami.

In [ ]:
# TODO: podmień województwo i liczbę ofert (pamiętaj o cudzysłowie, bo to tekst) i uruchom.
selected = df[(df["Województwo"] == "mazowieckie") & (df["Liczba ofert"] == "2")]
print("Znalezionych przetargów:", len(selected))
selected[["Gmina", "Liczba ofert", "Wartość umowy"]].head()

## 5. Sortowanie

`sort_values("kolumna")` układa wiersze od najmniejszej wartości.
Uwaga: `Data ogłoszenia` to tekst, więc pandas porównuje znak po znaku od lewej (najpierw dzień), a nie chronologicznie.

In [ ]:
# sort_values("Data ogłoszenia") układa wiersze po dacie; zapisujemy wynik do zmiennej, żeby zajrzeć do samych dat
sorted_by_date = df.sort_values("Data ogłoszenia")
# list() wokół head() wypisuje 5 pierwszych dat w jednej linii: wszystkie zaczynają się od dnia 01, a lata skaczą (2023, 2024, 2019),
# bo pandas porównuje tekst od lewej, nie datę
print(list(sorted_by_date["Data ogłoszenia"].head()))
# ascending=False odwraca kolejność (od "największej"); podwójne nawiasy = lista trzech kolumn do pokazania
# TODO: podmień "Data ogłoszenia" w sort_values na "Wartość umowy" i uruchom: czy na górze jest naprawdę największa kwota? (Nie, bo to tekst.)
df.sort_values("Data ogłoszenia", ascending=False)[["Gmina", "Data ogłoszenia", "Wartość umowy"]].head()

## 6. Braki i statystyki, czyli dlaczego trzeba czyścić (opcjonalnie, jeśli zostanie czas)

Puste pole pandas pokazuje jako `NaN` (brak wartości; w czystym Pythonie to samo znaczy `None`); `isna()` zaznacza braki, a `sum()` je zlicza.
`describe()` liczy statystyki, ale dla tekstu umie tylko zliczać wartości; średniej z `"1 234 567,89 zł"` nie policzy.

In [ ]:
# isna() = True tam, gdzie pole jest puste; sum() zlicza True w każdej kolumnie = liczba braków per kolumna
print(df.isna().sum())
# describe() na tekście: count (wypełnione), unique (różne), top (najczęstsza), freq (ile razy); żadnej średniej ani mediany
df.describe()

Gdybyśmy spróbowali policzyć cenę za Mg na surowych danych, dostalibyśmy błąd (dlatego tego nie uruchamiamy):

```python
df["Wartość umowy"] / df["Wolumen odpadów"]   # co by się stało: TypeError, nie da się dzielić tekstu przez tekst
```

Z tekstu `"1 250 000,50 zł"` Python nic nie policzy, dopóki nie zamienimy go na liczbę. Dokładnie to zrobimy w notebooku 06 (Pandas: czyszczenie danych): teksty zamienimy na liczby i daty.

## Zadania

Jeśli zabraknie czasu, sekcję 6 (Braki i statystyki, czyli dlaczego trzeba czyścić) i zadania robimy w domu. Rozwiązania są niżej, ale najpierw spróbuj sam.

1. Ile przetargów jest w województwie pomorskim? Zbuduj maskę i policz wiersze.
2. Ile przetargów w województwie śląskim było w trybie podstawowym? Dwa warunki z `&`. Wynik będzie mały, bo tryb podstawowy jest rzadki.
3. Pokaż 5 najczęstszych wykonawców. Potem pokaż 12 i poszukaj tej samej firmy dwa razy.
4. Posortuj tabelę po kolumnie `Gmina` i obejrzyj początek oraz koniec listy gmin. Co psuje kolejność?

In [ ]:
# Zadanie 1
# TODO: wzór: mask = df["Województwo"] == "małopolskie", potem len(df[mask]); podmień województwo na pomorskie.
# Podpowiedź: województwa są zapisane małymi literami: "pomorskie", nie "Pomorskie".

In [ ]:
# Zadanie 2
# TODO: dwa warunki w nawiasach połączone &: województwo "śląskie" i tryb "tryb podstawowy".
# Podpowiedź: dokładne nazwy trybów wypisze df["Tryb"].value_counts().

In [ ]:
# Zadanie 3
# TODO: df["Wykonawca"].value_counts() liczy, ile razy występuje każda firma; head() pokaże 5 pierwszych, head(12) dwanaście.
# Podpowiedź: patrz na początek nazw; spacja z przodu robi z jednej firmy dwie.

In [ ]:
# Zadanie 4
# TODO: df.sort_values("Gmina") układa tabelę po gminie; z wyniku wybierz kolumnę "Gmina" i pokaż jej .head(10) oraz .tail(5).
# Podpowiedź: list() wokół wyniku pokaże cudzysłowy i wtedy widać spacje.

## Rozwiązania

In [ ]:
# Rozwiązanie 1: maska dla jednego województwa i zliczenie wierszy z True
# zmienna mask już istnieje (małopolskie); nowa wartość zastępuje starą
mask = df["Województwo"] == "pomorskie"
print("Przetargów w woj. pomorskim:", len(df[mask]))

In [ ]:
# Rozwiązanie 2: dwa warunki, każdy w nawiasach, połączone & ("i")
mask_basic_mode = (df["Województwo"] == "śląskie") & (df["Tryb"] == "tryb podstawowy")
print("Śląskie i tryb podstawowy:", len(df[mask_basic_mode]))
# mały wynik to nie błąd: tryb podstawowy ma tylko 69 z 466 wierszy w całym pliku
# ten sam wzór z inną kolumną: drugi warunek to jedna oferta ("1" w cudzysłowie, bo tekst)
mask_single_bid = (df["Województwo"] == "śląskie") & (df["Liczba ofert"] == "1")
print("Śląskie i jedna oferta:", len(df[mask_single_bid]))
df[mask_single_bid][["Gmina", "Liczba ofert", "Wartość umowy"]].head()

In [ ]:
# Rozwiązanie 3: value_counts liczy, ile przetargów wygrał każdy wykonawca; head() = 5 najczęstszych
df["Wykonawca"].value_counts().head()

In [ ]:
# Rozwiązanie 3, ciąg dalszy: w dłuższej liście ta sama firma pojawia się drugi raz, tym razem ze spacją z przodu
# Dla pandas to dwie różne firmy; lekarstwem jest strip(), czyli obcięcie spacji z brzegów tekstu.
df["Wykonawca"].value_counts().head(12)

In [ ]:
# Rozwiązanie 4: sortujemy po gminie; list() pokazuje cudzysłowy, więc widać spacje
sorted_by_municipality = df.sort_values("Gmina")
# na początku lądują nazwy ze spacją z przodu, bo dla komputera spacja jest "mniejsza" niż każda litera
print(list(sorted_by_municipality["Gmina"].head(10)))
# na końcu lądują nazwy na Ż, bo polskie litery (Ł, Ś, Ż) komputer ustawia za wszystkimi łacińskimi, nawet za "z"
# "żory" jest za "Żory", bo dla komputera małe litery są "większe" niż wielkie (tak samo "bartoszyce" ląduje za wszystkimi nazwami pisanymi wielką literą)
# lekarstwem na wielkość liter jest title(), czyli wielka litera na początku każdego słowa
print(list(sorted_by_municipality["Gmina"].tail(5)))

**Podsumowanie**
- `pd.read_excel()` wczytuje Excel do tabeli `df`; `head()`, `shape`, `info()` to pierwszy rzut oka.
- `df["kolumna"]` to jedna kolumna; `value_counts()` liczy wartości, `nunique()` liczy różne wartości.
- Filtr to maska `True`/`False`: `df[df["Województwo"] == "śląskie"]`; dwa warunki łączymy `&` lub `|`, każdy w nawiasach.
- Surowe dane to sam tekst: daty i kwoty sortują się źle, `describe()` nie liczy średniej. Trzeba je wyczyścić.

Opcjonalna na zajęciach jest sekcja 6 (Braki i statystyki, czyli dlaczego trzeba czyścić): jeśli się nie zmieściła, doczytaj ją w domu. Zadania 1-4 też można zrobić w domu, rozwiązania są wyżej.

Dalej: notebook 06 (Pandas: czyszczenie danych), plik 06_pandas_czyszczenie.ipynb